# BirdCLEF 2026 — Multi-Resolution Ensemble — Inference Only (Pipeline 03)

This notebook performs **inference only** using three pre-trained models on 1s, 5s, and 30s windows.

**Instructions:**
- Attach your trained models (e.g. `best_model_1s.pth`, `best_model_5s.pth`, `best_model_30s.pth`).
- Update `CFG.MODEL_PATHS` to point to the checkpoints.


In [ ]:
import os, gc, math, glob, random, numpy as np, pandas as pd, soundfile as sf
from tqdm.auto import tqdm
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio.transforms as T
import timm
import warnings; warnings.filterwarnings('ignore')

class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')
    MODEL_PATHS = {
        1: 'best_model_1s.pth',
        5: 'best_model_5s.pth',
        30: 'best_model_30s.pth'
    }
    SR = 32000
    N_MELS, N_FFT, HOP_LENGTH, FMIN, FMAX = 128, 2048, 512, 20, 16000
    MODEL_NAME = 'tf_efficientnet_b0'
    ENSEMBLE_WEIGHTS = {1: 0.15, 5: 0.55, 30: 0.30}

CFG = Config()

sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
CFG.NUM_CLASSES = len(submission_labels)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, in_chans=3)
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
        self.head = nn.Linear(in_features, num_classes)
    def forward(self, x): return self.head(self.backbone(x))

models = {}
for w in [1, 5, 30]:
    m = BirdModel(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
    try:
        m.load_state_dict(torch.load(CFG.MODEL_PATHS[w], map_location=device))
        m.eval()
        models[w] = m
        print(f'Loaded model for {w}s')
    except:
        print(f'Failed to load {CFG.MODEL_PATHS[w]}')
        models[w] = m


In [ ]:
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg')) if os.path.exists(TEST_DIR) else []
if not test_files:
    print('FALLBACK: Using train soundscapes')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5]

mel_transform = T.MelSpectrogram(sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH, n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_preds, all_row_ids = [], []

for audio_path in tqdm(test_files):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    try:
        y, _ = sf.read(audio_path, always_2d=True)
        y = y.mean(axis=1)
    except: continue
    y_t = torch.tensor(y, dtype=torch.float32).to(device)
    total_samples = len(y_t)
    win_samples_5s = CFG.SR * 5
    n_segments = math.ceil(total_samples / win_samples_5s)
    for seg_idx in range(n_segments):
        end_time = (seg_idx + 1) * 5
        row_id = f'{filename}_{end_time}'
        
        # For 5s model:
        s5 = y_t[seg_idx * win_samples_5s : (seg_idx + 1) * win_samples_5s]
        if len(s5) < win_samples_5s: s5 = F.pad(s5, (0, win_samples_5s - len(s5)))
        
        # For 1s model: Split the 5s segment into five 1s segments
        s1_preds = []
        for i in range(5):
            s1 = s5[i * CFG.SR : (i + 1) * CFG.SR]
            with torch.no_grad():
                m = amplitude_to_db(mel_transform(s1))
                m = (m - m.min()) / (m.max() - m.min() + 1e-6)
                img = torch.stack([m, m, m]).unsqueeze(0)
                s1_preds.append(torch.sigmoid(models[1](img)).squeeze(0).cpu().numpy())
        p1 = np.mean(s1_preds, axis=0)
        
        # For 30s model: 30s window ending at end_time
        end_sample = end_time * CFG.SR
        start_sample = max(0, end_sample - 30 * CFG.SR)
        s30 = y_t[start_sample : end_sample]
        if len(s30) < 30 * CFG.SR: s30 = F.pad(s30, (30 * CFG.SR - len(s30), 0))
        
        with torch.no_grad():
            # 5s pred
            m5 = amplitude_to_db(mel_transform(s5))
            m5 = (m5 - m5.min()) / (m5.max() - m5.min() + 1e-6)
            p5 = torch.sigmoid(models[5](torch.stack([m5,m5,m5]).unsqueeze(0))).squeeze(0).cpu().numpy()
            
            # 30s pred
            m30 = amplitude_to_db(mel_transform(s30))
            m30 = (m30 - m30.min()) / (m30.max() - m30.min() + 1e-6)
            p30 = torch.sigmoid(models[30](torch.stack([m30,m30,m30]).unsqueeze(0))).squeeze(0).cpu().numpy()
            
        # Ensemble
        final_p = CFG.ENSEMBLE_WEIGHTS[1]*p1 + CFG.ENSEMBLE_WEIGHTS[5]*p5 + CFG.ENSEMBLE_WEIGHTS[30]*p30
        all_row_ids.append(row_id)
        all_preds.append(final_p)

sub_df = pd.DataFrame(all_preds, columns=submission_labels)
sub_df.insert(0, 'row_id', all_row_ids)
sub_df.to_csv('submission.csv', index=False)
print('Submission saved!')
